## Part 1 (initially part 3)

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import os
import json
from datetime import datetime

# ───────────────── INPUT / OUTPUT ─────────────────

INPUT_CSV = r"D:\Job\6.Gesis Student assistant\Data\8.Web Scraping-Gesis\Bernama_English\output_bernama\bernama_general.csv"

OUTPUT_CSV = r"D:\Job\6.Gesis Student assistant\Data\8.Web Scraping-Gesis\Bernama_English\output_bernama1\new_bernama_articles_output.csv"

# Folder where checkpoint CSVs will be saved (every 50 articles)
CHECKPOINT_DIR = r"D:\Job\6.Gesis Student assistant\Data\8.Web Scraping-Gesis\Bernama_English\output_bernama1\checkpoints"

KEYWORD = "general"

# ───────────────── RETRY CONFIG ─────────────────

MAX_RETRIES    = 3     # how many times to retry a failed request
RETRY_DELAY    = 5     # seconds to wait between retries
CHECKPOINT_EVERY = 50  # save a checkpoint CSV every N articles

# ───────────────── READ CSV ─────────────────

df = pd.read_csv(INPUT_CSV)

# ALL urls from CSV — no slicing
urls = df["url"].dropna().unique().tolist()

print(f"\nTotal URLs To Scrape: {len(urls)}")

# ───────────────── REQUEST HEADERS ─────────────────

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# ───────────────── STATUS CODE MESSAGES ─────────────────

# Map common HTTP codes to human-readable messages
STATUS_MESSAGES = {
    200: "OK",
    301: "Moved Permanently",
    302: "Found (Redirect)",
    403: "Forbidden",
    404: "Not Found",
    429: "Too Many Requests",
    500: "Internal Server Error",
    503: "Service Unavailable",
}

def get_status_msg(code):
    """Return a readable message for an HTTP status code."""
    return STATUS_MESSAGES.get(code, "Unknown Status")

# ───────────────── PARSER ─────────────────

def parse_article(html, url):

    soup = BeautifulSoup(html, "lxml")

    # ───────── TITLE ─────────

    title = ""

    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(" ", strip=True)

    elif soup.title:
        title = (
            soup.title.get_text(" ", strip=True)
            .replace("BERNAMA -", "")
            .strip()
        )

    # ───────── DATE + TIME ─────────

    published_date = ""
    published_time = ""

    meta_date = soup.find(
        "meta",
        {"property": "article:published_time"}
    )

    if meta_date:

        raw = meta_date.get("content", "").strip()

        parts = raw.split(" ")

        if len(parts) >= 3:

            published_date = parts[0]
            published_time = " ".join(parts[1:])

    # ───────── ARTICLE IMAGE ─────────

    article_img = ""

    topstory = soup.find("div", id="topstory")

    if topstory:

        carousel_inner = topstory.find(
            "div",
            class_="carousel-inner"
        )

        if carousel_inner:

            active_item = carousel_inner.find(
                "div",
                class_="carousel-item active"
            )

            if active_item:

                img = active_item.find("img")

                if img:

                    article_img = (
                        img.get("data-src")
                        or img.get("src")
                        or ""
                    ).strip()

    # fallback image
    if not article_img:

        meta_img = soup.find(
            "meta",
            {"property": "og:image"}
        )

        if meta_img:

            article_img = meta_img.get(
                "content",
                ""
            ).strip()

    # ───────── ARTICLE TEXT ─────────

    paragraphs = []

    for p in soup.find_all("p"):

        text = p.get_text(" ", strip=True)

        if not text:
            continue

        if len(text) < 40:
            continue

        bad_patterns = [
            "BERNAMA provides up-to-date",
            "Follow us on social media",
            "Facebook :",
            "Twitter :",
            "Instagram :",
            "TikTok :",
            "PORTAL KERJAYA BERNAMA",
            "Close"
        ]

        if any(x in text for x in bad_patterns):
            continue

        paragraphs.append(text)

    paragraphs = list(dict.fromkeys(paragraphs))

    article_text = "\n\n".join(paragraphs)

    return {
        "keyword": KEYWORD,
        "published_date": published_date,
        "published_time": published_time,
        "title": title,
        "article_text": article_text,
        "article_img": article_img,
        "url": url
    }

# ───────────────── CHECKPOINT SAVE ─────────────────

def save_checkpoint(articles, checkpoint_index):
    """Save a batch of articles (with all log columns) to a numbered checkpoint CSV."""

    # create folder if it doesn't exist
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"checkpoint_{checkpoint_index:04d}.csv"
    )

    pd.DataFrame(articles).to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"  [CHECKPOINT] Saved {len(articles)} articles → {checkpoint_path}")

# ───────────────── SCRIPT START TIME ─────────────────

script_start     = time.time()
script_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\nSCRIPT STARTED: {script_start_str}")

# ───────────────── FETCH LOOP ─────────────────

all_articles     = []   # all collected rows (article data + log data)
checkpoint_buffer = []  # holds rows since last checkpoint
checkpoint_index  = 1   # which checkpoint number we're on

total_urls = len(urls)

for idx, url in enumerate(urls, start=1):

    print("\n" + "=" * 70)
    print(f"[{idx}/{total_urls}] FETCHING")
    print(f"URL: {url}")

    # ── per-request tracking variables ──
    response          = None
    attempt           = 0
    request_start     = None
    request_start_str = ""
    request_end_str   = ""
    request_duration  = 0.0
    status_code       = ""
    status_msg        = ""
    redirected        = "No"
    redirect_hops     = ""          # e.g. "301 → url1 | 302 → url2"
    final_url         = url
    req_headers_str   = ""          # request headers as JSON string
    resp_headers_str  = ""          # response headers as JSON string
    wait_time         = 0.0
    total_attempts    = 0
    error_msg         = ""
    scrape_status     = "SKIPPED"   # SKIPPED / SUCCESS / ERROR

    while attempt < MAX_RETRIES:

        attempt += 1
        print(f"  ATTEMPT: {attempt}/{MAX_RETRIES}")

        # record start time for this individual request
        request_start     = time.time()
        request_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"  REQUEST START: {request_start_str}")

        try:

            response = requests.get(
                url,
                headers=headers,
                timeout=30,
                allow_redirects=True   # follow redirects automatically
            )

            # record end time right after response received
            request_end      = time.time()
            request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            request_duration = round(request_end - request_start, 2)

            print(f"  REQUEST END:   {request_end_str}")
            print(f"  REQUEST TIME:  {request_duration}s")

            # ── status code + message ──
            status_code = response.status_code
            status_msg  = get_status_msg(status_code)
            print(f"  STATUS: {status_code} — {status_msg}")

            # ── redirect detection ──
            # response.history holds every intermediate response before the final one
            if response.history:
                redirected   = f"Yes ({len(response.history)} hop(s))"
                hops_list    = [
                    f"{hop.status_code} → {hop.url}"
                    for hop in response.history
                ]
                redirect_hops = " | ".join(hops_list)
                final_url     = response.url
                print(f"  REDIRECTED: {redirected}")
                for hop_num, hop in enumerate(response.history, 1):
                    print(f"    hop {hop_num}: {hop.status_code} → {hop.url}")
                print(f"  FINAL URL: {final_url}")
            else:
                redirected = "No"
                print(f"  REDIRECTED: No")

            # ── capture request headers as JSON string for CSV ──
            req_headers_str  = json.dumps(dict(response.request.headers), ensure_ascii=False)

            # ── capture response headers as JSON string for CSV ──
            resp_headers_str = json.dumps(dict(response.headers), ensure_ascii=False)

            # ── print request headers ──
            print("  REQUEST HEADERS:")
            for k, v in response.request.headers.items():
                print(f"    {k}: {v}")

            # ── print response headers ──
            print("  RESPONSE HEADERS:")
            for k, v in response.headers.items():
                print(f"    {k}: {v}")

            # if successful, break out of retry loop
            if status_code == 200:
                break

            # if rate-limited or server error, wait before retrying
            if status_code in (429, 500, 503):
                print(f"  Retryable status {status_code}. Waiting {RETRY_DELAY}s before retry...")
                time.sleep(RETRY_DELAY)
            else:
                # non-retryable error (404, 403, etc.) — don't retry
                print(f"  Non-retryable status {status_code}. Skipping.")
                break

        except Exception as e:
            # network error, timeout, etc.
            request_end      = time.time()
            request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            request_duration = round(request_end - request_start, 2)
            error_msg        = str(e)
            scrape_status    = "ERROR"
            print(f"  ERROR on attempt {attempt}: {error_msg}")
            print(f"  REQUEST TIME (before error): {request_duration}s")

            if attempt < MAX_RETRIES:
                print(f"  Waiting {RETRY_DELAY}s before retry...")
                time.sleep(RETRY_DELAY)

    # save how many attempts were actually made
    total_attempts = attempt

    # ── parse if we got a valid 200 response ──
    if response is not None and response.status_code == 200:

        html = response.text

        soup = BeautifulSoup(html, "lxml")

        print(f"  TITLE CHECK: {soup.title}")
        print(f"  P TAGS: {len(soup.find_all('p'))}")

        article = parse_article(html, url)

        print(f"  TITLE:        {article['title'][:100]}")
        print(f"  DATE:         {article['published_date']}")
        print(f"  TIME:         {article['published_time']}")
        print(f"  IMAGE FOUND:  {'Yes' if article['article_img'] else 'No'}")
        print(f"  TEXT CHARS:   {len(article['article_text'])}")

        scrape_status = "SUCCESS"

    else:
        # fill article fields with blanks so the row still exists in CSV
        article = {
            "keyword":        KEYWORD,
            "published_date": "",
            "published_time": "",
            "title":          "",
            "article_text":   "",
            "article_img":    "",
            "url":            url
        }
        print(f"  SKIPPED (no valid response after {total_attempts} attempt(s))")

    # ── wait between requests ──
    wait_time = round(random.uniform(5, 10), 1)
    print(f"\n  WAITING {wait_time}s before next request...")
    time.sleep(wait_time)

    # ── merge article data + all log fields into one row ──
    row = {
        # ── original article fields ──
        **article,

        # ── request timing ──
        "log_request_start":    request_start_str,
        "log_request_end":      request_end_str,
        "log_request_duration_s": request_duration,

        # ── retry info ──
        "log_total_attempts":   total_attempts,
        "log_max_retries":      MAX_RETRIES,

        # ── wait time after this request ──
        "log_wait_time_s":      wait_time,

        # ── HTTP status ──
        "log_status_code":      status_code,
        "log_status_msg":       status_msg,

        # ── redirect info ──
        "log_redirected":       redirected,
        "log_redirect_hops":    redirect_hops,
        "log_final_url":        final_url,

        # ── headers (stored as JSON strings) ──
        "log_request_headers":  req_headers_str,
        "log_response_headers": resp_headers_str,

        # ── error message if any ──
        "log_error_msg":        error_msg,

        # ── overall result for this URL ──
        "log_scrape_status":    scrape_status,
    }

    all_articles.append(row)
    checkpoint_buffer.append(row)

    # ── checkpoint: save every N rows ──
    if len(checkpoint_buffer) >= CHECKPOINT_EVERY:
        save_checkpoint(checkpoint_buffer, checkpoint_index)
        checkpoint_index  += 1
        checkpoint_buffer  = []   # reset buffer after saving

# ── save any leftover rows that didn't fill a full checkpoint ──
if checkpoint_buffer:
    save_checkpoint(checkpoint_buffer, checkpoint_index)

# ───────────────── SAVE FINAL CSV ─────────────────

out_df = pd.DataFrame(all_articles)

out_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ───────────────── SCRIPT END TIME ─────────────────

script_end      = time.time()
script_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
total_duration  = round(script_end - script_start, 2)

print("\n" + "=" * 70)
print("SCRAPING COMPLETE")
print(f"SCRIPT STARTED:       {script_start_str}")
print(f"SCRIPT ENDED:         {script_end_str}")
print(f"TOTAL TIME:           {total_duration}s  ({round(total_duration/60, 2)} min)")
print(f"TOTAL ROWS SAVED:     {len(out_df)}")
print(f"  ↳ SUCCESS:          {(out_df['log_scrape_status'] == 'SUCCESS').sum()}")
print(f"  ↳ SKIPPED/ERROR:    {(out_df['log_scrape_status'] != 'SUCCESS').sum()}")
print(f"OUTPUT FILE:          {OUTPUT_CSV}")
print("=" * 70)


Total URLs To Scrape: 811

SCRIPT STARTED: 2026-05-30 23:17:31

[1/811] FETCHING
URL: https://www.bernama.com/en/news.php?id=2559131
  ATTEMPT: 1/3
  REQUEST START: 2026-05-30 23:17:31
  REQUEST END:   2026-05-30 23:17:32
  REQUEST TIME:  1.52s
  STATUS: 200 — OK
  REDIRECTED: No
  REQUEST HEADERS:
    User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36
    Accept-Encoding: gzip, deflate, br, zstd
    Accept: */*
    Connection: keep-alive
    Accept-Language: en-US,en;q=0.9
    Referer: https://www.google.com/
  RESPONSE HEADERS:
    Date: Sat, 30 May 2026 21:17:31 GMT
    Server: Apache
    X-Powered-By: PHP/8.2.30
    Expires: Thu, 19 Nov 1981 08:52:00 GMT
    Cache-Control: no-store, no-cache, must-revalidate
    Pragma: no-cache
    Set-Cookie: PHPSESSID=06n2628vov28lv9do4mc6r0v8s; path=/
    Strict-Transport-Security: max-age=31536000
    X-Frame-Options: SAMEORIGIN
    X-Content-Type-Options: nosniff
    Re